# Diagnostic Tests I — Exercises
### Applied Statistical Data Analysis — Prof. Dr. Kristyna Ters | MSc Finance | FHNW

---
> **Instructions:**
> - Work through the exercises in order — each builds on the previous one
> - Fill in your code in the cells marked with `# YOUR CODE HERE`
> - Answer written questions by double-clicking the markdown cell and editing it
> - Run cells with **Shift+Enter**
> - Solutions will be released after the submission deadline

In [ ]:
!pip install yfinance pandas-datareader statsmodels --quiet

import yfinance as yf
import pandas_datareader.data as web
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, het_white, het_goldfeldquandt, acorr_breusch_godfrey
from statsmodels.stats.stattools import durbin_watson
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.alpha':0.3, 'font.size':11
})
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'
GREY   = '#4B4B4B'; BLUE = '#0E75FE'; GREEN = '#0B7A3C'
print('✓ Libraries loaded.')

---
# Exercise 1 — Symptom or Not? (Warm-Up, No Code)

For each finding, state whether it points to **heteroskedasticity (H)**, **autocorrelation (A)**, or **neither (N)** — and which test you would run to confirm.

| # | Finding | H / A / N | Test |
|---|---------|-----------|------|
| a | The residual-vs-fitted scatter widens like a fan to the right | | |
| b | The residual plot over time shows long smooth swings above, then below zero | | |
| c | Calm months alternate with wildly volatile months in the residual series | | |
| d | The residuals look like white noise in every plot | | |
| e | DW = 0.31 in a regression of two price *levels* | | |
| f | The stock's beta estimate doubles when one outlier day is removed | | |

**Your answers** (double-click to edit):

| # | H / A / N | Test | One-line reason |
|---|-----------|------|-----------------|
| a | | | |
| b | | | |
| c | | | |
| d | | | |
| e | | | |
| f | | | |

---
# Exercise 2 — Estimate the CAPM for a Swiss Stock

Our diagnostic patient for Exercises 2–6: the **UBS CAPM** against the SMI.

$$r_{UBSG,t} = \beta_0 + \beta_1\, r_{SMI,t} + u_t$$

In [ ]:
STOCK, INDEX = 'UBSG.SW', '^SSMI'   # feel free to switch the stock

# YOUR CODE HERE
# 1. Download STOCK and INDEX 2020-01-01 to 2024-12-31 (auto_adjust=True), compute daily returns
#    Rename the two columns to 'UBS' and 'SMI' — the later exercises refer to them by those names:
#    ret = ret.rename(columns={STOCK: 'UBS', INDEX: 'SMI'})
# 2. Fit plain OLS: y = ret['UBS'], X = sm.add_constant(ret['SMI'])
# 3. Print n, beta_hat, its (non-robust) SE and t-statistic, and R²


---
# Exercise 3 — Eyes First: The Residual Plots

Produce the two standard residual plots for the UBS CAPM and read them.

**Written question:** which violation do the plots suggest — and why is that *expected* for daily return data?

In [ ]:
# YOUR CODE HERE
# 1. Scatter: residuals (y) vs. fitted values (x) — look for the fan
# 2. Line plot: residuals over time — look for volatility clusters
# (use capm.resid and capm.fittedvalues from Exercise 2)


---
# Exercise 4 — Goldfeld-Quandt by Hand

Run the GQ test on the UBS CAPM, ordering by $|r_{SMI}|$ and dropping the middle ~17% of the days.

**Written question:** state $H_0$, the two df, your GQ statistic and your decision. Then name the two *choices* the test forced you to make.

In [ ]:
# YOUR CODE HERE
# 1. Order the sample by ret['SMI'].abs() (ascending)
# 2. drop = round(0.17 * n); half = (n - drop) // 2 → low half / high half
# 3. Fit OLS on each half, get s² = ssr / df_resid for both
# 4. GQ = s²_high / s²_low; compare with stats.f.ppf(0.95, df_high, df_low)


---
# Exercise 5 — Breusch-Pagan by Hand, then One-Liners

(a) Run the BP **auxiliary regression** yourself: regress $\hat{u}^2$ on the regressors, compute $LM = n \cdot R^2_{aux}$, and decide against $\chi^2_1$ (crit 3.84).
(b) Confirm with `het_breuschpagan` and add `het_white`.

**Written question:** why does White's test have TWO degrees of freedom here?

In [ ]:
# YOUR CODE HERE
# (a) aux = sm.OLS(capm.resid**2, X_capm).fit();  LM = n * aux.rsquared
# (b) het_breuschpagan(capm.resid, capm.model.exog); het_white(...)


---
# Exercise 6 — Fix the Inference: Robust Standard Errors

Refit the UBS CAPM with **HC1** robust standard errors and compare beta, SE and t with the plain OLS fit.

**Written question:** the beta does not change at all — why exactly?

In [ ]:
# YOUR CODE HERE
# 1. capm_hc = sm.OLS(ret['UBS'], X_capm).fit(cov_type='HC1')
# 2. Build a small comparison DataFrame: beta_hat, SE, t for OLS vs. HC1


---
# Exercise 7 — A Levels Regression and Its Waves

Now the autocorrelation patient: the **mortgage pass-through in weekly levels** (FRED: `MORTGAGE30US` on `DGS10`, 2010–2024).

Estimate it, plot the residuals over time, and compute the **Durbin-Watson** statistic and the implied $\hat{\rho} = 1 - DW/2$.

In [ ]:
# YOUR CODE HERE
# 1. m30 = web.DataReader('MORTGAGE30US', 'fred', '2010-01-01', '2024-12-31')
#    y10 = web.DataReader('DGS10', 'fred', '2010-01-01', '2024-12-31')
# 2. Merge weekly: lvl = m30.join(y10.resample('W-THU').mean(), how='inner').dropna()
# 3. OLS: mort on constant + y10; plot residuals over time
# 4. durbin_watson(mort.resid) → implied rho


---
# Exercise 8 — Breusch-Godfrey: Test the Memory Formally

Run BG with $p = 4$ lags on the levels regression — once **by hand** (auxiliary regression) and once with `acorr_breusch_godfrey`.

**Written question:** why 4 lags — and what are the df of the $\chi^2$?

In [ ]:
# YOUR CODE HERE
# By hand: regress û_t on [y10, û_{t-1..t-4}] (dropna!), LM = (n−p)·R²_aux, χ²(4) crit 9.49
# One line: acorr_breusch_godfrey(mort, nlags=4)


---
# Exercise 9 — Newey-West: Honest Standard Errors

(a) Compute the rule-of-thumb bandwidth $L \approx 0.75\,n^{1/3}$ for the mortgage sample.
(b) Refit with `cov_type='HAC'` and compare SE and t with plain OLS.

**Written question:** your sample has ~750 weeks. Roughly how large would $n$ have to be for the rule of thumb to allow **twice** as many lags?

In [ ]:
# YOUR CODE HERE
# L = ceil(0.75 * n ** (1/3))
# mort_nw = sm.OLS(...).fit(cov_type='HAC', cov_kwds={'maxlags': L})
# compare beta / SE / t : OLS vs Newey-West


---
# Exercise 10 — The Better Fix: Dynamics + Information Criteria

(a) Estimate the dynamic model $mort_t = \alpha + \sum_{j=1}^{p} \varphi_j\, mort_{t-j} + \theta\, y10_t + u_t$ for $p = 0, \dots, 4$ and tabulate **AIC and BIC** (Brooks' formulas: $\ln(\hat{\sigma}^2) + 2k/T$ and $\ln(\hat{\sigma}^2) + (k/T)\ln T$).
(b) Pick $p$ by the BIC minimum, re-estimate, and report: the adjustment coefficient, the short-run effect $\theta$, and the **long-run pass-through** $\theta / (1 - \sum\varphi_j)$.
(c) Re-run Breusch-Godfrey on the chosen model.

**Written question:** why must the final autocorrelation check use BG and **not** Durbin-Watson?

In [ ]:
# YOUR CODE HERE
# (a) loop p = 0..4: build lagged columns with .shift(j), dropna, OLS;
#     sigma2 = ssr/T, k = df_model + 1;  AIC = log(sigma2) + 2k/T;  BIC = log(sigma2) + k/T*log(T)
# (b) p* = BIC-minimum; refit; long-run = theta / (1 - sum of phi_j)
# (c) acorr_breusch_godfrey(m_dyn, nlags=4)


---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*